# TA-05: Klasifikasi Churn Pelanggan Telekomunikasi dengan Decision Tree
**Nama:** Irfan Dwi Darmawan | **NIM:** 301240034 | **Kelas:** 5 A

Dataset: [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn). Target: `Churn`.

## 1. Memuat Data

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, roc_curve, roc_auc_score)
warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9, "legend.fontsize": 8})
SEED = 42
os.makedirs("model", exist_ok=True); os.makedirs("static", exist_ok=True)

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. EDA dan Pra-pemrosesan

In [2]:
print(df.dtypes.value_counts())
print("\nNilai kosong (NaN):", int(df.isna().sum().sum()))
blank = df["TotalCharges"].astype(str).str.strip() == ""
print("TotalCharges berisi spasi kosong:", int(blank.sum()))
print("Tenure pada baris tersebut:", df.loc[blank, "tenure"].unique())
print("\nDistribusi target:"); print(df["Churn"].value_counts()); print(df["Churn"].value_counts(normalize=True).round(3))

str        18
int64       2
float64     1
Name: count, dtype: int64

Nilai kosong (NaN): 0
TotalCharges berisi spasi kosong: 11
Tenure pada baris tersebut: [0]

Distribusi target:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


Sebelas nilai kosong pada `TotalCharges` semuanya berasal dari pelanggan dengan `tenure = 0` (pelanggan baru yang belum pernah ditagih), sehingga nilainya diisi **0**, bukan median, karena secara logika total tagihan mereka memang nol.

In [3]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
sns.countplot(x="Churn", data=df, ax=ax[0]); ax[0].set_title("Distribusi target")
for a, col in zip(ax[1:], ["Contract", "InternetService"]):
    r = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
    a.bar(r.index, r.values, color="#4c78a8"); a.set_title(f"Persentase churn per {col}"); a.set_ylabel("% churn")
    a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig("static/eda1.jpg", dpi=200); plt.show()
print(df.groupby("Contract")["Churn"].apply(lambda s: round((s == "Yes").mean() * 100, 1)).to_dict())
print(df.groupby("InternetService")["Churn"].apply(lambda s: round((s == "Yes").mean() * 100, 1)).to_dict())

{'Month-to-month': 42.7, 'One year': 11.3, 'Two year': 2.8}
{'DSL': 19.0, 'Fiber optic': 41.9, 'No': 7.4}


In [4]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
sns.boxplot(x="Churn", y="tenure", data=df, ax=ax[0]); ax[0].set_title("Tenure (bulan)")
sns.boxplot(x="Churn", y="MonthlyCharges", data=df, ax=ax[1]); ax[1].set_title("Tagihan bulanan")
plt.tight_layout(); plt.savefig("static/eda2.jpg", dpi=200); plt.show()
print(df.groupby("Churn")[["tenure", "MonthlyCharges"]].median())

       tenure  MonthlyCharges
Churn                        
No       38.0          64.425
Yes      10.0          79.650

In [5]:
data = df.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce").fillna(0.0)
data = data.drop(columns=["customerID"])
data["Churn"] = (data["Churn"] == "Yes").astype(int)

# kolom biner Yes/No (dan gender) -> 0/1
for c in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
    data[c] = (data[c] == "Yes").astype(int)
data["gender"] = (data["gender"] == "Male").astype(int)   # 1 = Male

# kolom kategorikal multi-nilai -> one-hot
multi = ["MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
         "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaymentMethod"]
data = pd.get_dummies(data, columns=multi, dtype=int)

X = data.drop(columns="Churn"); y = data["Churn"]
print("Jumlah fitur setelah encoding:", X.shape[1]); print("Nilai kosong:", int(X.isna().sum().sum()))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(X_train.shape, X_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))

Jumlah fitur setelah encoding: 40
Nilai kosong: 0
(5634, 40) (1409, 40) 0.265 0.265


## 3. Model A (tanpa batasan) dan Model B (di-pruning)

In [6]:
model_a = DecisionTreeClassifier(max_depth=None, random_state=SEED).fit(X_train, y_train)
print("Model A -> kedalaman:", model_a.get_depth(), "| jumlah daun:", model_a.get_n_leaves())
print("Akurasi latih:", round(model_a.score(X_train, y_train), 4), "| Akurasi uji:", round(model_a.score(X_test, y_test), 4))

Model A -> kedalaman: 22 | jumlah daun: 1109
Akurasi latih: 0.998 | Akurasi uji: 0.7253


Hyperparameter Model B dipilih lewat *5-fold cross-validation* **hanya pada data latih**, bukan pada data uji. Metriknya ROC-AUC karena tidak bergantung pada ambang 0,5 dan cocok untuk kelas yang tidak seimbang. Dari semua kombinasi dipilih pohon **paling sederhana** (kedalaman terkecil) yang skor CV-nya masih dalam satu simpangan baku dari skor terbaik (*one-standard-error rule*), agar pohon tetap mudah dibaca.

In [7]:
grid = GridSearchCV(DecisionTreeClassifier(random_state=SEED),
                    {"max_depth": [2, 3, 4, 5, 6, 7, 8], "min_samples_leaf": [1, 10, 30, 50]},
                    cv=5, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
res = pd.DataFrame(grid.cv_results_)
res = res.rename(columns={"param_max_depth": "max_depth", "param_min_samples_leaf": "min_samples_leaf"})
best = res.loc[res["mean_test_score"].idxmax()]
thr = best["mean_test_score"] - best["std_test_score"]
print("Terbaik mutlak:", int(best["max_depth"]), int(best["min_samples_leaf"]), "AUC CV =", round(best["mean_test_score"], 4), "| std =", round(best["std_test_score"], 4))
cand = res[res["mean_test_score"] >= thr].sort_values(["max_depth", "min_samples_leaf"], ascending=[True, False])
pick = cand.iloc[0]
best_params = {"max_depth": int(pick["max_depth"]), "min_samples_leaf": int(pick["min_samples_leaf"])}
print("Ambang (1 SE):", round(thr, 4), "| Dipilih:", best_params, "| AUC CV =", round(pick["mean_test_score"], 4))
print(res.sort_values("mean_test_score", ascending=False)[["max_depth", "min_samples_leaf", "mean_test_score", "std_test_score"]].head(8).round(4).to_string())

Terbaik mutlak: 6 50 AUC CV = 0.8308 | std = 0.0174
Ambang (1 SE): 0.8135 | Dipilih: {'max_depth': 3, 'min_samples_leaf': 50} | AUC CV = 0.8187
    max_depth  min_samples_leaf  mean_test_score  std_test_score
19          6                50           0.8308          0.0174
15          5                50           0.8294          0.0175
27          8                50           0.8294          0.0176
23          7                50           0.8292          0.0183
11          4                50           0.8283          0.0119
10          4                30           0.8275          0.0110
18          6                30           0.8272          0.0165
8           4                 1           0.8271          0.0117


In [8]:
# kurva overfitting: akurasi latih vs validasi (CV) terhadap max_depth
depths = list(range(1, 21)); tr, cv = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    tr.append(m.fit(X_train, y_train).score(X_train, y_train))
    cv.append(cross_val_score(m, X_train, y_train, cv=5, scoring="accuracy").mean())
plt.figure(figsize=(6, 3.6)); plt.plot(depths, tr, "o-", ms=3, label="Akurasi latih"); plt.plot(depths, cv, "s-", ms=3, label="Akurasi validasi (CV)")
plt.title("Efek max_depth terhadap overfitting"); plt.xlabel("max_depth"); plt.ylabel("Akurasi"); plt.legend(); plt.grid(alpha=.4)
plt.tight_layout(); plt.savefig("static/depth_curve.jpg", dpi=200); plt.show()
print({d: (round(a, 3), round(b, 3)) for d, a, b in zip(depths, tr, cv) if d in (1, 3, 4, 5, 6, 8, 12, 20)})

{1: (0.735, np.float64(0.735)), 3: (0.791, np.float64(0.788)), 4: (0.797, np.float64(0.787)), 5: (0.804, np.float64(0.786)), 6: (0.81, np.float64(0.787)), 8: (0.837, np.float64(0.774)), 12: (0.925, np.float64(0.734)), 20: (0.997, np.float64(0.722))}


In [9]:
model_b = DecisionTreeClassifier(random_state=SEED, **best_params).fit(X_train, y_train)
print("Model B -> max_depth:", model_b.max_depth, "| min_samples_leaf:", model_b.min_samples_leaf,
      "| kedalaman:", model_b.get_depth(), "| jumlah daun:", model_b.get_n_leaves())

Model B -> max_depth: 3 | min_samples_leaf: 50 | kedalaman: 3 | jumlah daun: 8


## 4. Evaluasi dan Perbandingan

In [10]:
def evaluate(m, name):
    p_tr, p_te = m.predict(X_train), m.predict(X_test); proba = m.predict_proba(X_test)[:, 1]
    r = {"model": name, "depth": int(m.get_depth()), "leaves": int(m.get_n_leaves()),
         "acc_train": accuracy_score(y_train, p_tr), "acc_test": accuracy_score(y_test, p_te),
         "precision": precision_score(y_test, p_te), "recall": recall_score(y_test, p_te),
         "f1": f1_score(y_test, p_te), "auc": roc_auc_score(y_test, proba),
         "cm": confusion_matrix(y_test, p_te).tolist()}
    print(f"===== {name} ====="); print(confusion_matrix(y_test, p_te))
    print(classification_report(y_test, p_te, target_names=["Bertahan", "Churn"], digits=3)); return r
ra = evaluate(model_a, "Model A (tanpa batas)"); rb = evaluate(model_b, "Model B (pruned)")
print(pd.DataFrame([ra, rb]).drop(columns="cm").round(4).to_string())

===== Model A (tanpa batas) =====
[[834 201]
 [186 188]]
              precision    recall  f1-score   support

    Bertahan      0.818     0.806     0.812      1035
       Churn      0.483     0.503     0.493       374

    accuracy                          0.725      1409
   macro avg      0.650     0.654     0.652      1409
weighted avg      0.729     0.725     0.727      1409

===== Model B (pruned) =====
[[970  65]
 [242 132]]
              precision    recall  f1-score   support

    Bertahan      0.800     0.937     0.863      1035
       Churn      0.670     0.353     0.462       374

    accuracy                          0.782      1409
   macro avg      0.735     0.645     0.663      1409
weighted avg      0.766     0.782     0.757      1409

                   model  depth  leaves  acc_train  acc_test  precision  recall      f1     auc
0  Model A (tanpa batas)     22    1109     0.9980    0.7253     0.4833  0.5027  0.4928  0.6540
1       Model B (pruned)      3       8     0

In [11]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
for a, r in zip(ax, [ra, rb]):
    sns.heatmap(np.array(r["cm"]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Bertahan", "Churn"], yticklabels=["Bertahan", "Churn"])
    a.set_title(r["model"]); a.set_xlabel("Prediksi"); a.set_ylabel("Sebenarnya")
plt.tight_layout(); plt.savefig("static/cm.jpg", dpi=200); plt.show()

In [12]:
plt.figure(figsize=(5, 4))
for m, n, c in [(model_a, "Model A", "tab:red"), (model_b, "Model B", "tab:blue")]:
    pr = m.predict_proba(X_test)[:, 1]; f, t, _ = roc_curve(y_test, pr)
    plt.plot(f, t, color=c, label=f"{n} (AUC = {roc_auc_score(y_test, pr):.2f})")
plt.plot([0, 1], [0, 1], "k--", label="Tebakan acak"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Kurva ROC"); plt.legend(); plt.grid(alpha=.4); plt.tight_layout(); plt.savefig("static/roc.jpg", dpi=200); plt.show()

## 5. Visualisasi dan Interpretasi Model B

In [13]:
plt.figure(figsize=(16, 7))
plot_tree(model_b, feature_names=list(X.columns), class_names=["Bertahan", "Churn"], filled=True, rounded=True,
          max_depth=3, fontsize=8)
plt.title("Pohon Keputusan Model B (max_depth=3)"); plt.tight_layout(); plt.savefig("static/tree.jpg", dpi=170); plt.show()

In [14]:
rules = export_text(model_b, feature_names=list(X.columns), max_depth=4, show_weights=True)
print(rules[:4000])

|--- Contract_Month-to-month <= 0.50
|   |--- MonthlyCharges <= 93.67
|   |   |--- OnlineSecurity_No <= 0.50
|   |   |   |--- weights: [1469.00, 36.00] class: 0
|   |   |--- OnlineSecurity_No >  0.50
|   |   |   |--- weights: [360.00, 39.00] class: 0
|   |--- MonthlyCharges >  93.67
|   |   |--- Contract_Two year <= 0.50
|   |   |   |--- weights: [255.00, 70.00] class: 0
|   |   |--- Contract_Two year >  0.50
|   |   |   |--- weights: [279.00, 24.00] class: 0
|--- Contract_Month-to-month >  0.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- tenure <= 3.50
|   |   |   |--- weights: [266.00, 202.00] class: 0
|   |   |--- tenure >  3.50
|   |   |   |--- weights: [743.00, 184.00] class: 0
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- tenure <= 13.50
|   |   |   |--- weights: [221.00, 541.00] class: 1
|   |   |--- tenure >  13.50
|   |   |   |--- weights: [546.00, 399.00] class: 0



In [15]:
imp = pd.Series(model_b.feature_importances_, index=X.columns).sort_values(ascending=False)
top = imp.head(8)
print(imp.head(10).round(4)); print("Fitur dengan importance > 0:", int((imp > 0).sum()))
plt.figure(figsize=(6, 3.6)); sns.barplot(x=top.values, y=top.index, palette="viridis")
plt.title("Feature importance Model B (8 teratas)"); plt.xlabel("Importance"); plt.ylabel("")
plt.tight_layout(); plt.savefig("static/importance.jpg", dpi=200); plt.show()

Contract_Month-to-month        0.6022
InternetService_Fiber optic    0.1913
tenure                         0.1721
MonthlyCharges                 0.0191
Contract_Two year              0.0097
OnlineSecurity_No              0.0057
gender                         0.0000
SeniorCitizen                  0.0000
PaperlessBilling               0.0000
PhoneService                   0.0000
dtype: float64
Fitur dengan importance > 0: 6


## 6. Menyimpan Model

In [16]:
joblib.dump(model_b, "model/decision_tree.joblib")
joblib.dump(model_a, "model/decision_tree_overfit.joblib")
t = model_b.tree_
tree = {"feature": t.feature.tolist(), "threshold": t.threshold.tolist(), "left": t.children_left.tolist(),
        "right": t.children_right.tolist(), "value": t.value[:, 0, :].tolist(), "n_samples": t.n_node_samples.tolist()}
json.dump({"features": list(X.columns), "tree": tree}, open("model/tree.json", "w"))
json.dump({"model_a": ra, "model_b": rb, "best_params": best_params,
           "cv_auc": float(pick["mean_test_score"]), "cv_auc_best": float(best["mean_test_score"]), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
           "churn_rate": float(y.mean()),
           "ranges": {c: [float(pd.to_numeric(df[c], errors="coerce").min()), float(pd.to_numeric(df[c], errors="coerce").max())] for c in ["tenure", "MonthlyCharges", "TotalCharges"]},
           "importance": {k: float(v) for k, v in imp.head(10).items()}},
          open("model/metrics.json", "w"), indent=2)
X_test.assign(Churn=y_test).to_csv("model/test_set.csv", index=False)
print("Tersimpan.")

Tersimpan.
